# 03 — Train a 5-second JSBSim skill Transformer

This notebook consumes the canonical Parquet trajectories and label map produced by
`02_generate_jsbsim_skill_dataset.ipynb`. Complete flights are assigned to an
approximately **0.6:0.2:0.2 train:test:validation split**, stratified by the complete
set of tactical skills demonstrated in each flight.

Only `MODEL_FEATURE_COLUMNS` are model inputs. Commanded skills, native actions, and
other privileged columns are targets or audit metadata only. Transition-spanning
windows are excluded by default; set `BVR_KEEP_MIXED_WINDOWS=1` to label them by their
final sample.

The Transformer is trained on the training split. In accordance with this experiment's
selection protocol, test loss controls checkpointing and early stopping. The validation
split remains untouched until the selected checkpoint receives its final evaluation.
MLflow records configuration, per-epoch metrics, the checkpoint, and the complete
reproducibility bundle.

## 1. Imports, reproducibility, and MLflow configuration

The dataset path exactly matches notebook 02's default output. Environment variables
allow short, reproducible experiments without editing cells. CPU is the safe default;
set `BVR_TRAIN_DEVICE=cuda` only after verifying the local CUDA installation.

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as pads
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from torch.utils.data import DataLoader, TensorDataset

from bvr_behavior_prediction.data.observable_columns import MODEL_FEATURE_COLUMNS
from bvr_behavior_prediction.data.privileged_columns import PRIVILEGED_COLUMNS

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
DATASET_DIR = Path(os.getenv(
    "BVR_TRAIN_DATASET",
    REPO_ROOT / "artifacts/datasets/bvr_f16_1v1_jsbsim_skills_v001",
))
OUTPUT_DIR = Path(os.getenv(
    "BVR_CLASSIFIER_OUTPUT", REPO_ROOT / "artifacts/models/jsbsim_skill_transformer_v001"
))
MLFLOW_TRACKING_URI = os.getenv(
    "BVR_MLFLOW_TRACKING_URI", (REPO_ROOT / "artifacts/mlruns").resolve().as_uri()
)
MLFLOW_EXPERIMENT = os.getenv("BVR_MLFLOW_EXPERIMENT", "jsbsim-skill-transformer")
WINDOW_S = 5.0
STRIDE_S = float(os.getenv("BVR_WINDOW_STRIDE_S", "1.0"))
KEEP_MIXED_WINDOWS = os.getenv("BVR_KEEP_MIXED_WINDOWS", "0") == "1"
BATCH_SIZE = int(os.getenv("BVR_BATCH_SIZE", "64"))
MICRO_BATCH_SIZE = int(os.getenv("BVR_MICRO_BATCH_SIZE", str(BATCH_SIZE)))
ACCUMULATION_STEPS = math.ceil(BATCH_SIZE / MICRO_BATCH_SIZE)
GRADIENT_CHECKPOINTING = os.getenv("BVR_GRADIENT_CHECKPOINTING", "1") == "1"
PREPROCESS_CHUNK_WINDOWS = int(os.getenv("BVR_PREPROCESS_CHUNK_WINDOWS", "8192"))
ARROW_BATCH_ROWS = int(os.getenv("BVR_ARROW_BATCH_ROWS", "65536"))
KEEP_WINDOW_CACHE = os.getenv("BVR_KEEP_WINDOW_CACHE", "0") == "1"
DATALOADER_WORKERS = int(os.getenv("BVR_DATALOADER_WORKERS", "0"))
USE_AMP = os.getenv("BVR_MIXED_PRECISION", "1") == "1"
TORCH_THREADS = int(os.getenv("BVR_TORCH_THREADS", str(min(4, os.cpu_count() or 1))))
REQUESTED_DEVICE = os.getenv("BVR_TRAIN_DEVICE", "cpu").strip().lower()
EPOCHS = int(os.getenv("BVR_TRAIN_EPOCHS", "30"))
PATIENCE = int(os.getenv("BVR_EARLY_STOPPING_PATIENCE", "6"))
LEARNING_RATE = float(os.getenv("BVR_LEARNING_RATE", "0.001"))
D_MODEL = int(os.getenv("BVR_TRANSFORMER_D_MODEL", "128"))
NHEAD = int(os.getenv("BVR_TRANSFORMER_HEADS", "8"))
NUM_LAYERS = int(os.getenv("BVR_TRANSFORMER_LAYERS", "3"))
DROPOUT = float(os.getenv("BVR_TRANSFORMER_DROPOUT", "0.2"))
SEED = int(os.getenv("BVR_TRAIN_SEED", "20260911"))
SPLIT_FRACTIONS = {"train": 0.60, "test": 0.20, "validation": 0.20}

if min(BATCH_SIZE, MICRO_BATCH_SIZE, PREPROCESS_CHUNK_WINDOWS, ARROW_BATCH_ROWS) < 1 or DATALOADER_WORKERS < 0:
    raise ValueError("Batch size must be positive and data-loader workers cannot be negative")
if BATCH_SIZE % MICRO_BATCH_SIZE or MICRO_BATCH_SIZE > BATCH_SIZE:
    raise ValueError("BVR_MICRO_BATCH_SIZE must divide BVR_BATCH_SIZE and cannot exceed it")
if TORCH_THREADS < 1:
    raise ValueError("BVR_TORCH_THREADS must be at least 1")
if D_MODEL % NHEAD:
    raise ValueError("BVR_TRANSFORMER_D_MODEL must be divisible by BVR_TRANSFORMER_HEADS")
if REQUESTED_DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(
        f"BVR_TRAIN_DEVICE={REQUESTED_DEVICE!r} requires CUDA, but this PyTorch "
        "installation cannot access it. Use BVR_TRAIN_DEVICE=cpu or install a "
        "CUDA-compatible PyTorch build."
    )

torch.set_num_threads(TORCH_THREADS)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if REQUESTED_DEVICE.startswith("cuda"):
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device(REQUESTED_DEVICE)
AMP_ENABLED = USE_AMP and DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    # TF32 speeds supported matrix multiplications without increasing memory use.
    torch.set_float32_matmul_precision("high")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print({"dataset": str(DATASET_DIR), "device": str(DEVICE), "seed": SEED,
       "mlflow_tracking_uri": mlflow.get_tracking_uri()})

## 2. Scan dataset metadata without materializing the trajectories

PyArrow streams bounded record batches and projects only the columns needed by each pass.
The first pass retains just one compact row per episode for stratification and cadence
validation. It never creates a dataset-sized Arrow table or pandas frame. A later pass
reads model features directly into disk-backed NumPy arrays, so peak RAM is bounded by
one Arrow batch, one episode, and the configured preprocessing chunk.


In [ ]:
shards = sorted((DATASET_DIR / "trajectories").glob("*.parquet"))
label_map_path = DATASET_DIR / "label_map.json"
if not shards or not label_map_path.exists():
    raise FileNotFoundError(
        f"Expected trajectories/*.parquet and label_map.json under {DATASET_DIR}. "
        "Run notebooks/02_generate_jsbsim_skill_dataset.ipynb first or set BVR_TRAIN_DATASET."
    )

LABELS = json.loads(label_map_path.read_text())["tactical"]
label_to_index = {label: index for index, label in enumerate(LABELS)}
required_columns = ["episode_id", "time_s", "tactical_label", *MODEL_FEATURE_COLUMNS]
trajectory_dataset = pads.dataset(shards, format="parquet")
missing = set(required_columns).difference(trajectory_dataset.schema.names)
assert not missing, f"Missing required columns: {sorted(missing)}"
assert set(MODEL_FEATURE_COLUMNS).isdisjoint(PRIVILEGED_COLUMNS)


def iter_episodes(columns):
    """Yield contiguous episodes while holding at most one scan batch plus one episode."""
    scanner = trajectory_dataset.scanner(
        columns=columns, batch_size=ARROW_BATCH_ROWS, use_threads=True,
        batch_readahead=4, fragment_readahead=2,
    )
    pending_id, pending = None, {column: [] for column in columns}
    try:
        for batch in scanner.to_batches():
            arrays = {
                column: batch.column(column).to_numpy(zero_copy_only=False)
                for column in columns
            }
            ids = arrays["episode_id"]
            boundaries = np.r_[0, np.flatnonzero(ids[1:] != ids[:-1]) + 1, len(ids)]
            for left, right in zip(boundaries[:-1], boundaries[1:]):
                episode_id = ids[left]
                if pending_id is not None and episode_id != pending_id:
                    yield pending_id, {
                        column: np.concatenate(parts) if len(parts) > 1 else parts[0]
                        for column, parts in pending.items()
                    }
                    pending = {column: [] for column in columns}
                pending_id = episode_id
                for column in columns:
                    pending[column].append(arrays[column][left:right])
        if pending_id is not None:
            yield pending_id, {
                column: np.concatenate(parts) if len(parts) > 1 else parts[0]
                for column, parts in pending.items()
            }
    except pa.ArrowKeyError as error:
        raise RuntimeError(
            "PyArrow's legacy extension registry is incompatible with this pandas session. "
            "Install the project dependencies (which require pyarrow>=14.0.1), restart the "
            "notebook kernel, and run all cells again."
        ) from error


episode_rows, sample_count, cadence = [], 0, None
for episode_id, episode in iter_episodes(["episode_id", "time_s", "tactical_label"]):
    times = episode["time_s"]
    deltas = np.diff(times)
    episode_dt = float(np.median(deltas))
    assert episode_dt > 0 and np.allclose(deltas, episode_dt, atol=1e-6), "Irregular sample cadence"
    cadence = episode_dt if cadence is None else cadence
    assert math.isclose(episode_dt, cadence, abs_tol=1e-6), "Inconsistent episode cadence"
    skills = tuple(sorted(set(episode["tactical_label"])))
    unknown = set(skills).difference(label_to_index)
    assert not unknown, f"Labels absent from label_map.json: {sorted(unknown)}"
    episode_rows.append({"episode_id": episode_id, "skills": skills, "samples": len(times)})
    sample_count += len(times)

SAMPLE_DT_S = cadence
WINDOW_SAMPLES = int(round(WINDOW_S / SAMPLE_DT_S))
STRIDE_SAMPLES = max(1, int(round(STRIDE_S / SAMPLE_DT_S)))
assert WINDOW_SAMPLES >= 2
print(f"Scanned {sample_count:,} samples from {len(episode_rows):,} episodes")


## 3. Stratify complete episodes 60:20:20 by demonstrated skills

A flight can demonstrate more than one skill. Its stratum is therefore the sorted set
of all tactical labels appearing in that episode, rather than only its first or most
frequent label. A 60/40 stratified split is followed by an equal stratified division of
the remainder. This preserves joint skill combinations while keeping every flight—and
all overlapping windows from it—in exactly one split.

Every demonstrated-skill combination needs at least five episodes to be represented in
all three partitions. The production dataset from notebook 02 readily satisfies this;
a deliberately tiny smoke dataset fails with an actionable message rather than silently
falling back to an unstratified split.

In [ ]:
episode_skills = pd.DataFrame.from_records(episode_rows)
episode_skills["stratum"] = episode_skills["skills"].map("|".join)
stratum_counts = episode_skills["stratum"].value_counts()
if len(episode_skills) < 5 or (stratum_counts < 5).any():
    rare = stratum_counts[stratum_counts < 5].to_dict()
    raise ValueError(
        "Stratified 60:20:20 splitting requires at least five flights for every "
        f"demonstrated-skill combination; insufficient strata: {rare}. Generate more flights."
    )

train_episodes, selection_episodes = train_test_split(
    episode_skills,
    test_size=SPLIT_FRACTIONS["test"] + SPLIT_FRACTIONS["validation"],
    random_state=SEED,
    shuffle=True,
    stratify=episode_skills["stratum"],
)
test_episodes, validation_episodes = train_test_split(
    selection_episodes,
    test_size=0.5,
    random_state=SEED,
    shuffle=True,
    stratify=selection_episodes["stratum"],
)
split_tables = {
    "train": train_episodes,
    "test": test_episodes,
    "validation": validation_episodes,
}
split_ids = {name: table["episode_id"].tolist() for name, table in split_tables.items()}
all_split_ids = [set(ids) for ids in split_ids.values()]
assert all(all_split_ids) and not any(
    all_split_ids[i] & all_split_ids[j]
    for i in range(len(all_split_ids)) for j in range(i + 1, len(all_split_ids))
)
assert set().union(*all_split_ids) == set(episode_skills["episode_id"])

split_audit = pd.concat([
    table.assign(split=name).explode("skills")
    for name, table in split_tables.items()
])
split_summary = pd.crosstab(split_audit["skills"], split_audit["split"])
split_summary.loc["TOTAL EPISODES"] = {
    name: len(table) for name, table in split_tables.items()
}
print({name: round(len(ids) / len(episode_skills), 4) for name, ids in split_ids.items()})
display(split_summary)

## 4. Stream five-second windows into disk-backed arrays

After splitting, a label-only streaming pass counts admissible windows. A feature pass
then writes directly into `.npy` memory maps instead of keeping the source trajectories
and all split arrays in RAM. Mixed-window detection is vectorized with a cumulative
change count. The maps are demand-paged by the OS during training and can therefore be
much larger than host memory without an allocation spike.


In [ ]:
episode_to_split = {
    episode_id: split_name for split_name, ids in split_ids.items() for episode_id in ids
}
window_counts = {name: 0 for name in split_ids}
mixed_counts = {name: 0 for name in split_ids}


def window_starts(targets):
    starts = np.arange(0, len(targets) - WINDOW_SAMPLES + 1, STRIDE_SAMPLES)
    if not len(starts):
        return starts, np.empty(0, dtype=bool)
    changes = np.r_[0, np.cumsum(targets[1:] != targets[:-1])]
    mixed = changes[starts + WINDOW_SAMPLES - 1] != changes[starts]
    return starts, mixed


for episode_id, episode in iter_episodes(["episode_id", "tactical_label"]):
    split_name = episode_to_split[episode_id]
    starts, mixed = window_starts(episode["tactical_label"])
    mixed_counts[split_name] += int(mixed.sum())
    window_counts[split_name] += len(starts) if KEEP_MIXED_WINDOWS else int((~mixed).sum())
if not all(window_counts.values()):
    raise ValueError("No windows were produced; check episode duration and filtering")

CACHE_DIR = Path(os.getenv(
    "BVR_WINDOW_CACHE", OUTPUT_DIR.parent / f".{OUTPUT_DIR.name}_window_cache"
))
shutil.rmtree(CACHE_DIR, ignore_errors=True)
CACHE_DIR.mkdir(parents=True)
raw = {}
for split_name, count in window_counts.items():
    raw[split_name] = {
        "x": np.lib.format.open_memmap(
            CACHE_DIR / f"{split_name}_x.npy", mode="w+", dtype=np.float32,
            shape=(count, WINDOW_SAMPLES, len(MODEL_FEATURE_COLUMNS)),
        ),
        "y": np.lib.format.open_memmap(
            CACHE_DIR / f"{split_name}_y.npy", mode="w+", dtype=np.int64, shape=(count,)
        ),
        "episode_index": np.lib.format.open_memmap(
            CACHE_DIR / f"{split_name}_episode.npy", mode="w+", dtype=np.int32, shape=(count,)
        ),
        "start_time_s": np.lib.format.open_memmap(
            CACHE_DIR / f"{split_name}_start.npy", mode="w+", dtype=np.float32, shape=(count,)
        ),
        "end_time_s": np.lib.format.open_memmap(
            CACHE_DIR / f"{split_name}_end.npy", mode="w+", dtype=np.float32, shape=(count,)
        ),
        "mixed_skill": np.lib.format.open_memmap(
            CACHE_DIR / f"{split_name}_mixed.npy", mode="w+", dtype=np.bool_, shape=(count,)
        ),
    }

episode_ids = episode_skills["episode_id"].tolist()
episode_indices = {episode_id: index for index, episode_id in enumerate(episode_ids)}
write_offsets = {name: 0 for name in split_ids}
for episode_id, episode in iter_episodes(required_columns):
    split_name = episode_to_split[episode_id]
    starts, mixed = window_starts(episode["tactical_label"])
    keep = np.ones(len(starts), dtype=bool) if KEEP_MIXED_WINDOWS else ~mixed
    starts, mixed = starts[keep], mixed[keep]
    if not len(starts):
        continue
    destination = raw[split_name]
    left, right = write_offsets[split_name], write_offsets[split_name] + len(starts)
    values = np.column_stack([episode[column] for column in MODEL_FEATURE_COLUMNS]).astype(
        np.float32, copy=False
    )
    # This bounded loop writes each view straight to disk; no dataset-sized np.stack is created.
    for output_index, start in enumerate(starts, left):
        destination["x"][output_index] = values[start:start + WINDOW_SAMPLES]
    targets = episode["tactical_label"]
    destination["y"][left:right] = [label_to_index[targets[s + WINDOW_SAMPLES - 1]] for s in starts]
    destination["episode_index"][left:right] = episode_indices[episode_id]
    times = episode["time_s"]
    destination["start_time_s"][left:right] = times[starts]
    destination["end_time_s"][left:right] = times[starts + WINDOW_SAMPLES - 1]
    destination["mixed_skill"][left:right] = mixed
    write_offsets[split_name] = right

assert write_offsets == window_counts
for split_name in split_ids:
    for array in raw[split_name].values():
        array.flush()
    print(split_name, {"episodes": len(split_ids[split_name]),
                       "windows": window_counts[split_name],
                       "excluded_mixed": mixed_counts[split_name]})
# Complete episode assignment already proves that overlapping windows cannot leak.
assert len(episode_to_split) == len(set(episode_to_split)) == len(episode_ids)
gc.collect()


## 5. Incremental, in-place preprocessing

`StandardScaler.partial_fit` sees training data in bounded chunks, preventing its
float64 accumulators and temporary reductions from scaling with the full window array.
Normalization also proceeds chunk by chunk and in place. DataLoader workers do not copy
the memory maps, and CUDA pins only individual micro-batches for asynchronous transfer.


In [ ]:
scaler = StandardScaler(copy=False)
train_x = raw["train"]["x"]
for left in range(0, len(train_x), PREPROCESS_CHUNK_WINDOWS):
    chunk = train_x[left:left + PREPROCESS_CHUNK_WINDOWS]
    scaler.partial_fit(chunk.reshape(-1, chunk.shape[-1]))
normalization_mean = scaler.mean_.astype(np.float32)
normalization_scale = scaler.scale_.astype(np.float32)

loaders = {}
for split_name, values in raw.items():
    x = values["x"]
    for left in range(0, len(x), PREPROCESS_CHUNK_WINDOWS):
        chunk = x[left:left + PREPROCESS_CHUNK_WINDOWS]
        x[left:left + len(chunk)] -= normalization_mean
        x[left:left + len(chunk)] /= normalization_scale
    x.flush()
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(values["y"]))
    generator = torch.Generator().manual_seed(SEED)
    loaders[split_name] = DataLoader(
        dataset, batch_size=MICRO_BATCH_SIZE, shuffle=split_name == "train",
        generator=generator if split_name == "train" else None,
        num_workers=DATALOADER_WORKERS, pin_memory=DEVICE.type == "cuda",
        persistent_workers=DATALOADER_WORKERS > 0,
    )
print("Train tensor:", tuple(loaders["train"].dataset.tensors[0].shape),
      "effective batch:", MICRO_BATCH_SIZE * ACCUMULATION_STEPS)


## 6. Memory-efficient Transformer training tracked by MLflow

CUDA uses automatic mixed precision, fused AdamW, TF32, pinned asynchronous transfers,
and PyTorch's optimized attention kernels. Training uses a configurable micro-batch and
gradient accumulation to preserve the requested effective batch size. Activation
checkpointing (on by default) recomputes encoder layers during backward, trading modest
compute for a large reduction in peak activation memory. Disable it with
`BVR_GRADIENT_CHECKPOINTING=0` when throughput matters more than memory, or tune
`BVR_MICRO_BATCH_SIZE` independently of `BVR_BATCH_SIZE`.

After every epoch, **test loss** controls checkpointing and early stopping. MLflow
receives parameters and train/test metrics for every epoch.


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_length, dropout):
        super().__init__()
        positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
        frequencies = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10_000.0) / d_model)
        )
        encoding = torch.zeros(max_length, d_model)
        encoding[:, 0::2] = torch.sin(positions * frequencies)
        encoding[:, 1::2] = torch.cos(positions * frequencies)
        self.register_buffer("encoding", encoding.unsqueeze(0), persistent=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, sequence):
        return self.dropout(sequence + self.encoding[:, :sequence.size(1)])


class SkillTransformer(nn.Module):
    def __init__(self, input_dim, class_count, window_samples, d_model=128,
                 nhead=8, layers=3, dropout=0.2):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.positions = SinusoidalPositionalEncoding(d_model, window_samples, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4 * d_model,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=layers, norm=nn.LayerNorm(d_model)
        )
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, class_count))

    def forward(self, sequence):
        encoded = self.positions(self.input_projection(sequence))
        if self.training and GRADIENT_CHECKPOINTING:
            for layer in self.encoder.layers:
                encoded = torch_checkpoint(layer, encoded, use_reentrant=False)
            if self.encoder.norm is not None:
                encoded = self.encoder.norm(encoded)
        else:
            encoded = self.encoder(encoded)
        return self.classifier(encoded.mean(dim=1))


model_config = {
    "input_dim": len(MODEL_FEATURE_COLUMNS), "class_count": len(LABELS),
    "window_samples": WINDOW_SAMPLES, "d_model": D_MODEL, "nhead": NHEAD,
    "layers": NUM_LAYERS, "dropout": DROPOUT,
}
model = SkillTransformer(**model_config).to(DEVICE)
counts = np.bincount(raw["train"]["y"], minlength=len(LABELS))
weights = counts.sum() / (len(LABELS) * np.maximum(counts, 1))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4, fused=DEVICE.type == "cuda"
)
grad_scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


def run_epoch(loader, training=False):
    model.train(training)
    total_loss = total_correct = total = 0
    if training:
        optimizer.zero_grad(set_to_none=True)
    for batch_index, (features, target) in enumerate(loader):
        features = features.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(training), torch.autocast(
            device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED
        ):
            logits = model(features)
            loss = criterion(logits, target)
        if training:
            grad_scaler.scale(loss / ACCUMULATION_STEPS).backward()
            update = (batch_index + 1) % ACCUMULATION_STEPS == 0 or batch_index + 1 == len(loader)
            if update:
                grad_scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                grad_scaler.step(optimizer)
                grad_scaler.update()
                optimizer.zero_grad(set_to_none=True)
        total_loss += loss.detach().item() * len(target)
        total_correct += (logits.detach().argmax(1) == target).sum().item()
        total += len(target)
    return {"loss": total_loss / total, "accuracy": total_correct / total}


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.pt"
history, best_test_loss, stale_epochs = [], float("inf"), 0
mlflow_params = {
    **model_config, "architecture": "transformer_encoder", "batch_size": BATCH_SIZE,
    "micro_batch_size": MICRO_BATCH_SIZE, "accumulation_steps": ACCUMULATION_STEPS,
    "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "epochs": EPOCHS, "patience": PATIENCE, "learning_rate": LEARNING_RATE,
    "seed": SEED, "device": str(DEVICE), "mixed_precision": AMP_ENABLED,
    "dataloader_workers": DATALOADER_WORKERS, "window_s": WINDOW_S, "stride_s": STRIDE_S,
    **{f"split_{name}": fraction for name, fraction in SPLIT_FRACTIONS.items()},
}

with mlflow.start_run(run_name=f"transformer-seed-{SEED}") as active_run:
    run_id = active_run.info.run_id
    mlflow.log_params(mlflow_params)
    mlflow.set_tags({"dataset": DATASET_DIR.name, "checkpoint_selection_split": "test"})
    for epoch in range(1, EPOCHS + 1):
        train_metrics = run_epoch(loaders["train"], training=True)
        test_metrics = run_epoch(loaders["test"])
        row = {"epoch": epoch,
               **{f"train_{key}": value for key, value in train_metrics.items()},
               **{f"test_{key}": value for key, value in test_metrics.items()}}
        history.append(row)
        mlflow.log_metrics({key: value for key, value in row.items() if key != "epoch"}, step=epoch)
        print(f"{epoch:02d} train loss={train_metrics['loss']:.4f} "
              f"acc={train_metrics['accuracy']:.3f} test loss={test_metrics['loss']:.4f} "
              f"acc={test_metrics['accuracy']:.3f}")
        if test_metrics["loss"] < best_test_loss - 1e-4:
            best_test_loss = test_metrics["loss"]
            stale_epochs = 0
            torch.save({
                "epoch": epoch, "test_loss": best_test_loss,
                "model_state_dict": model.state_dict(), "model_config": model_config,
            }, CHECKPOINT_PATH)
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                print("Early stopping on test loss")
                break

    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    mlflow.log_metric("best_test_loss", checkpoint["test_loss"])
    mlflow.log_metric("best_epoch", checkpoint["epoch"])
    mlflow.log_artifact(str(CHECKPOINT_PATH), artifact_path="checkpoints")

history = pd.DataFrame(history)
history.plot(x="epoch", y=["train_loss", "test_loss"], grid=True,
             title="MLflow-tracked learning curves")
plt.show()
print({"mlflow_run_id": run_id, "selected_epoch": checkpoint["epoch"]})

## 7. Final evaluation on the untouched validation set

Validation is not used for scaling, optimization, checkpoint selection, or early
stopping. It provides the final class-wise report for the test-selected checkpoint.

In [ ]:
def predict(loader):
    model.eval()
    actual, predicted = [], []
    with torch.inference_mode():
        for features, target in loader:
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                logits = model(features.to(DEVICE, non_blocking=True))
            actual.extend(target.numpy().tolist())
            predicted.extend(logits.argmax(1).cpu().numpy().tolist())
    return np.asarray(actual), np.asarray(predicted)


y_validation, y_pred = predict(loaders["validation"])
validation_accuracy = float((y_validation == y_pred).mean())
report = classification_report(
    y_validation, y_pred, labels=np.arange(len(LABELS)), target_names=LABELS,
    zero_division=0, digits=3, output_dict=True,
)
print(classification_report(
    y_validation, y_pred, labels=np.arange(len(LABELS)), target_names=LABELS,
    zero_division=0, digits=3,
))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_validation, y_pred, labels=np.arange(len(LABELS)), display_labels=LABELS,
    normalize="true", xticks_rotation=45, cmap="Blues", ax=ax,
)
ax.set_title("Untouched validation confusion matrix (row normalized)")
plt.tight_layout()
plt.show()

## 8. Save and log the reproducible bundle

The bundle includes the test-selected Transformer, training-only normalization,
stratified episode assignments, class and feature order, window configuration, history,
and validation report. The same files are attached to the active MLflow run.

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(), "model_config": model_config,
    "feature_columns": list(MODEL_FEATURE_COLUMNS), "labels": LABELS,
    "window_samples": WINDOW_SAMPLES, "sample_dt_s": SAMPLE_DT_S,
    "selected_epoch": checkpoint["epoch"], "selection_test_loss": checkpoint["test_loss"],
}, OUTPUT_DIR / "model.pt")
(OUTPUT_DIR / "preprocessing.json").write_text(json.dumps({
    "feature_columns": list(MODEL_FEATURE_COLUMNS),
    "mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
}, indent=2))
(OUTPUT_DIR / "split.json").write_text(json.dumps({
    "seed": SEED, "fractions": SPLIT_FRACTIONS, "stratification": "demonstrated_skill_set",
    "episode_ids": split_ids, "window_s": WINDOW_S, "stride_s": STRIDE_S,
    "keep_mixed_windows": KEEP_MIXED_WINDOWS,
}, indent=2))
history.to_csv(OUTPUT_DIR / "history.csv", index=False)
(OUTPUT_DIR / "validation_report.json").write_text(json.dumps(report, indent=2))
def write_window_metadata(split_name, values):
    """Write metadata incrementally without constructing a multi-million-row DataFrame."""
    import pyarrow.parquet as pq

    destination = OUTPUT_DIR / f"{split_name}_windows.parquet"
    writer = None
    try:
        for left in range(0, len(values["y"]), PREPROCESS_CHUNK_WINDOWS):
            right = min(left + PREPROCESS_CHUNK_WINDOWS, len(values["y"]))
            indices = values["episode_index"][left:right]
            table = pa.table({
                "episode_id": [episode_ids[index] for index in indices],
                "start_time_s": values["start_time_s"][left:right],
                "end_time_s": values["end_time_s"][left:right],
                "mixed_skill": values["mixed_skill"][left:right],
                "label": [LABELS[index] for index in values["y"][left:right]],
            })
            writer = writer or pq.ParquetWriter(destination, table.schema, compression="zstd")
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()


for split_name, values in raw.items():
    write_window_metadata(split_name, values)

# Do not upload the potentially enormous temporary memory maps as MLflow artifacts.
bundle_files = ["model.pt", "preprocessing.json", "split.json", "history.csv",
                "validation_report.json", *[f"{name}_windows.parquet" for name in raw]]
with mlflow.start_run(run_id=run_id):
    mlflow.log_metric("validation_accuracy", validation_accuracy)
    for filename in bundle_files:
        mlflow.log_artifact(str(OUTPUT_DIR / filename), artifact_path="training_bundle")
print("Saved training bundle to", OUTPUT_DIR.resolve())

if not KEEP_WINDOW_CACHE:
    # Drop every tensor/memmap view before removing backing files (also works on Windows).
    del loaders, dataset, train_x, x, chunk, values, raw
    gc.collect()
    shutil.rmtree(CACHE_DIR)
    print("Removed temporary window cache", CACHE_DIR.resolve())


## Interpretation notes

* Results are episode-held-out, not independent random-window performance.
* The test partition is intentionally a **development selection set** in this protocol;
  report the untouched validation metrics as the final generalization estimate.
* Per-class metrics matter because aggregate accuracy can hide weak rare-skill behavior.
* Mixed windows are better suited to a future transition or multi-label model.
* Synthetic performance does not establish real-world tactical generalization.